<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/Phase6_Monitoring-DriftDetection-and-overnance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Develop an Audit Log System

### Subtask:
Implement a simple audit logging mechanism that records prediction requests, actual predictions, and the model version used for each inference. This can be stored in a data structure like a list of dictionaries or a pandas DataFrame for later analysis.


In [1]:
# This cell imports all necessary libraries for data manipulation, machine learning, and data validation.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from pydantic import BaseModel, ValidationError, field_validator
import datetime

In [2]:
# This cell handles mounting Google Drive, loading the raw patient, visit, and billing datasets, and then merging them into a single DataFrame (`df_merged`).
# Upload file to google.colab
import io
from google.colab import files, drive

drive.mount('/content/drive')

# comment this code, if files already uploaded.
# uploaded = files.upload()

# Load and read patients, visits & billing csv
patients_df = pd.read_csv('/content/drive/MyDrive/files/capstone/patients.csv')
# patients_df.head()

visits_df = pd.read_csv('/content/drive/MyDrive/files/capstone/visits.csv')
# visits_df.head()

billing_df = pd.read_csv('/content/drive/MyDrive/files/capstone/billing.csv')
# billing_df.head()

# Merge dataframe
df_merged = pd.merge(patients_df, visits_df, on='patient_id', how = 'inner')
df_merged = pd.merge(df_merged, billing_df, on='visit_id', how = 'inner')
#df_merged.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# This cell defines the Pydantic `DataValidator` class, which is used for real-time validation of input data features, ensuring data quality and adherence to specified rules (e.g., age range, positive billing, valid approval ratio, and binary chronic flag).
# 1. Real-time Data Validation Schema
class DataValidator(BaseModel):
    age: int
    billed_amount: float
    approval_ratio: float
    chronic_flag: int

    @field_validator('age')
    @classmethod
    def age_within_range(cls, v):
        if not (0 <= v <= 120):
            raise ValueError('Age must be between 0 and 120')
        return v

    @field_validator('billed_amount')
    @classmethod
    def positive_billing(cls, v):
        if np.isnan(v):
            raise ValueError('Billed amount cannot be NaN')
        if v < 0:
            raise ValueError('Billed amount cannot be negative')
        return v

    @field_validator('approval_ratio')
    @classmethod
    def ratio_limit(cls, v):
        if np.isnan(v):
            raise ValueError('Approval ratio cannot be NaN')
        if not (0.0 <= v <= 1.0):
            raise ValueError('Approval ratio must be between 0 and 1')
        return v

    @field_validator('chronic_flag')
    @classmethod
    def chronic_flag_values(cls, v):
        if v not in [0, 1]:
            raise ValueError('Chronic flag must be 0 or 1')
        return v

## Implementation of the core monitoring logic:
* `check_feature_drift` for detecting shifts in input data features,
* `check_prediction_drift` for identifying changes in model prediction distributions
* `AuditLog` class for recording inference details.



In [4]:
# 2. Batch Drift Detection Logic
def check_feature_drift(reference_df, current_df, threshold=0.1):
    """
    Compares the mean of key features between the training
    reference and the current production window.
    """
    drift_report = {}
    features_to_track = ['billed_amount', 'approval_ratio', 'age']

    for feature in features_to_track:
        ref_mean = reference_df[feature].mean()
        curr_mean = current_df[feature].mean()

        # Handle cases where ref_mean might be zero to avoid division by zero
        if ref_mean == 0:
            drift = abs(curr_mean - ref_mean) # If ref_mean is 0, absolute diff is the drift
        else:
            drift = abs(curr_mean - ref_mean) / ref_mean

        drift_report[feature] = {
            "drift_score": round(drift, 4),
            "alert": drift > threshold
        }
    return drift_report

# 3. Prediction Drift Detection Logic
def check_prediction_drift(reference_predictions, current_predictions, threshold=0.1):
    """
    Compares the distribution of model predictions between a reference dataset
    and a current dataset by checking the mean.
    """
    ref_mean = np.mean(reference_predictions)
    curr_mean = np.mean(current_predictions)

    # Calculate drift score
    # Handle division by zero if reference mean is zero
    if ref_mean == 0:
        drift_score = abs(curr_mean - ref_mean) # Or handle as a specific alert if appropriate
    else:
        drift_score = abs(curr_mean - ref_mean) / ref_mean

    drift_report = {
        "drift_score": round(drift_score, 4),
        "alert": drift_score > threshold
    }
    return drift_report

# 4. Audit Log System
class AuditLog:
    def __init__(self):
        self.log_entries = []

    def log_prediction(self, request_data, prediction, model_version):
        """
        Logs a prediction request, its result, and the model version used.
        """
        log_entry = {
            'timestamp': datetime.datetime.now().isoformat(),
            'request_data': request_data,
            'prediction': prediction,
            'model_version': model_version
        }
        self.log_entries.append(log_entry)

    def get_log_as_dataframe(self):
        """
        Returns the audit log as a pandas DataFrame.
        """
        return pd.DataFrame(self.log_entries) if self.log_entries else pd.DataFrame()

### Integrated Monitoring Report Function
* `run_monitoring_report` function, that orchestrates the execution of data validation, feature drift detection, prediction drift detection, and audit logging to generate a consolidated monitoring report.



In [5]:
# 5. Integrated Monitoring Report Function
def run_monitoring_report(reference_df, current_df, reference_predictions, current_predictions, model_version, drift_threshold=0.1):
    """
    Orchestrates the execution of data validation, drift detection, and audit logging,
    generating a consolidated monitoring report.
    """
    print(f"\n--- Generating Monitoring Report (Model Version: {model_version}) ---")

    # 5.1. Data Validation
    print("\n1. Data Validation Check:")
    valid_records = 0
    invalid_records = 0
    validation_errors = []

    for index, row in current_df.iterrows():
        try:
            DataValidator(**row.to_dict())
            valid_records += 1
        except ValidationError as e:
            invalid_records += 1
            validation_errors.append(f"Record {index}: {e.errors()}")

    print(f"   Total records: {len(current_df)}")
    print(f"   Valid records: {valid_records}")
    print(f"   Invalid records: {invalid_records}")
    if validation_errors:
        print("   Validation Errors detected:")
        for error in validation_errors[:3]: # Print first 3 errors for brevity
            print(f"      - {error}")
        if len(validation_errors) > 3:
            print(f"      ... and {len(validation_errors) - 3} more errors.")
    else:
        print("   No validation errors detected.")

    # 5.2. Feature Drift Detection
    print("\n2. Feature Drift Detection:")
    feature_drift_report = check_feature_drift(reference_df, current_df, threshold=drift_threshold)
    for feature, report in feature_drift_report.items():
        alert_status = "ALERT" if report['alert'] else "OK"
        print(f"   - {feature}: Drift Score = {report['drift_score']}, Status: {alert_status}")
    if any(report['alert'] for report in feature_drift_report.values()):
        print("   Feature drift detected in one or more features!")
    else:
        print("   No significant feature drift detected.")

    # 5.3. Prediction Drift Detection
    print("\n3. Prediction Drift Detection:")
    prediction_drift_report = check_prediction_drift(reference_predictions, current_predictions, threshold=drift_threshold)
    alert_status = "ALERT" if prediction_drift_report['alert'] else "OK"
    print(f"   - Prediction Drift Score = {prediction_drift_report['drift_score']}, Status: {alert_status}")
    if prediction_drift_report['alert']:
        print("   Prediction drift detected!")
    else:
        print("   No significant prediction drift detected.")

    # 5.4. Audit Log Simulation
    print("\n4. Audit Log Summary:")
    audit_log = AuditLog()
    # Simulate logging a few predictions
    for i in range(5):
        sample_request = {'id': i, 'features': current_df.iloc[i].to_dict()}
        sample_prediction = np.random.rand()
        audit_log.log_prediction(sample_request, sample_prediction, model_version)

    print(f"   Number of prediction entries logged: {len(audit_log.log_entries)}")
    # audit_df = audit_log.get_log_as_dataframe() # Can retrieve full log if needed
    # print(audit_df.head())

    print("\n--- Monitoring Report Generation Complete ---")

### Testing of drift detection
* Prepare the reference and current DataFrames for testing by:
  1. calculating 'approval_ratio' for the merged data,
  2. generating synthetic 'current_df' with simulated drift and validation errors, and defining sample predictions.
  3. reference_df : df_merged is used as reference data
  4. current_df: synthetic data is generated to showcase drift.
* Finally, it executes the integrated monitoring report.


In [6]:
# Calculate approval_ratio for df_merged (to be used as reference_df)
# Handle potential division by zero and NaNs for billed_amount
df_merged['approval_ratio'] = df_merged.apply(
    lambda row: row['approved_amount'] / row['billed_amount'] if row['billed_amount'] != 0 else 0,
    axis=1
)
# Ensure approval_ratio is within 0 and 1, clipping values if necessary
df_merged['approval_ratio'] = df_merged['approval_ratio'].clip(0, 1)

# Use df_merged as the reference DataFrame
reference_df = df_merged.copy() # Use a copy to avoid modifying the original df_merged

# Generate current_df with drift and errors
num_samples = 100

# Get statistics from df_merged to inform synthetic data generation
age_mean = df_merged['age'].mean()
age_std = df_merged['age'].std()
billed_amount_mean = df_merged['billed_amount'].mean()
billed_amount_std = df_merged['billed_amount'].std()
approval_ratio_mean = df_merged['approval_ratio'].mean()
approval_ratio_std = df_merged['approval_ratio'].std()
chronic_flag_prop_1 = df_merged['chronic_flag'].value_counts(normalize=True).get(1, 0)

# Generate current_df with some drift
current_data = {
    'age': np.random.normal(age_mean + 5, age_std * 1.2, num_samples).astype(int).clip(0, 120), # Shifted mean, wider std
    'billed_amount': np.random.normal(billed_amount_mean * 0.8, billed_amount_std * 1.5, num_samples), # Shifted mean, wider std
    'approval_ratio': np.random.normal(approval_ratio_mean * 0.7, approval_ratio_std * 1.5, num_samples), # Shifted mean, wider std
    'chronic_flag': np.random.choice([0, 1], num_samples, p=[1 - (chronic_flag_prop_1 * 1.5).clip(0,1), (chronic_flag_prop_1 * 1.5).clip(0,1)])
}

current_df = pd.DataFrame(current_data)

# Introduce some explicit errors for validation testing on current_df
# Billed amount: negative, NaN
current_df.loc[0, 'billed_amount'] = -10.0
current_df.loc[1, 'billed_amount'] = np.nan
# Approval ratio: out of range, NaN
current_df.loc[2, 'approval_ratio'] = 1.5
current_df.loc[3, 'approval_ratio'] = -0.2
current_df.loc[4, 'approval_ratio'] = np.nan
# Chronic flag: invalid values
current_df.loc[5, 'chronic_flag'] = 2
current_df.loc[6, 'chronic_flag'] = 3

# Sample Predictions for drift detection
reference_predictions = np.random.uniform(0.1, 0.9, num_samples) # Baseline predictions
current_predictions = np.random.uniform(0.4, 1.2, num_samples) # Shifted predictions

# Model Version
current_model_version = "v1.2.0"

# Run the monitoring report
run_monitoring_report(reference_df, current_df, reference_predictions, current_predictions, current_model_version, drift_threshold=0.1)

print("Integrated monitoring report script created and executed.")


--- Generating Monitoring Report (Model Version: v1.2.0) ---

1. Data Validation Check:
   Total records: 100
   Valid records: 49
   Invalid records: 51
   Validation Errors detected:
      - Record 0: [{'type': 'value_error', 'loc': ('billed_amount',), 'msg': 'Value error, Billed amount cannot be negative', 'input': -10.0, 'ctx': {'error': ValueError('Billed amount cannot be negative')}, 'url': 'https://errors.pydantic.dev/2.12/v/value_error'}]
      - Record 1: [{'type': 'value_error', 'loc': ('billed_amount',), 'msg': 'Value error, Billed amount cannot be NaN', 'input': nan, 'ctx': {'error': ValueError('Billed amount cannot be NaN')}, 'url': 'https://errors.pydantic.dev/2.12/v/value_error'}, {'type': 'value_error', 'loc': ('approval_ratio',), 'msg': 'Value error, Approval ratio must be between 0 and 1', 'input': -0.2813301464966418, 'ctx': {'error': ValueError('Approval ratio must be between 0 and 1')}, 'url': 'https://errors.pydantic.dev/2.12/v/value_error'}]
      - Record 2: [{